# Part 1: Introduction to Machine Learning in Big Data

## Introduction to Machine Learning in Big Data:
Machine learning involves teaching computers to learn from data, identify patterns,
and make decisions with minimal human intervention. With big data, these processes
involve larger datasets and require more advanced techniques to manage and analyze data effectively.

### Key ML Concepts:
- **Supervised Learning:** In supervised learning, the algorithm learns from labeled data, i.e., data that is already tagged with the correct answer. It learns to predict the output from the input data.
- **Unsupervised Learning:** Unsupervised learning allows the algorithm to act on its own to discover patterns in the data. It explores the data and identifies hidden structures without any prior information about labels.
- **Model Evaluation:** Model evaluation is essential to assess the performance of a machine learning model. Metrics such as accuracy, precision, recall, and F1 score are commonly used to measure how well the model performs on unseen data.

### Python Libraries for ML:
- `scikit-learn`: Scikit-learn is a comprehensive library for machine learning in Python. It provides simple and efficient tools for data mining and data analysis, built on NumPy, SciPy, and Matplotlib.
- `pyspark MLlib`: MLlib is Apache Spark's scalable machine learning library. It provides a wide range of algorithms and utilities for large-scale machine learning tasks, including classification, regression, clustering, and collaborative filtering.

### Challenges of ML with Big Data:
- **Scalability:** Dealing with large volumes of data requires algorithms and techniques that can scale efficiently to handle the increased computational load.
- **Data Quality:** Big data often comes with challenges related to data quality, such as missing values, inconsistencies, and noise. Managing and cleaning such data is crucial for building accurate machine learning models.
- **Computational Complexity:** Processing and analyzing large datasets can be computationally intensive and time-consuming. Efficient algorithms and distributed computing frameworks like Apache Spark are necessary to tackle these challenges effectively.


# Part 2: Follow Me - Building a Machine Learning Model with scikit-learn


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
# Step 1: Loading Data
# Load the healthcare dataset from a CSV file into a pandas DataFrame.
healthcare_df = pd.read_csv('healthcare.csv')
healthcare_df.tail()

,Age,Systolic_BP,Diastolic_BP,Cholesterol,Disease_Presence
9995,45,156,105,227,1
9996,41,157,86,208,0
9997,50,129,118,253,1
9998,62,146,90,231,1
9999,29,111,106,267,0


In [3]:
healthcare_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Age               10000 non-null  int64
 1   Systolic_BP       10000 non-null  int64
 2   Diastolic_BP      10000 non-null  int64
 3   Cholesterol       10000 non-null  int64
 4   Disease_Presence  10000 non-null  int64
dtypes: int64(5)
memory usage: 390.8 KB


In [4]:
healthcare_df.describe()

,Age,Systolic_BP,Diastolic_BP,Cholesterol,Disease_Presence
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,44.620400,144.554200,94.483000,224.574700,0.514200
std,14.343489,20.219143,14.385256,43.007985,0.499823
min,20.000000,110.000000,70.000000,150.000000,0.000000
25%,32.000000,127.000000,82.000000,187.000000,0.000000
50%,45.000000,145.000000,94.000000,225.000000,1.000000
75%,57.000000,162.000000,107.000000,262.000000,1.000000
max,69.000000,179.000000,119.000000,299.000000,1.000000


In [6]:
# Step 2: Data Preprocessing
# Check for missing values and handle them as necessary (omitted here for brevity).
# Assuming the target variable 'Disease_Presence' is categorical and needs encoding if it's in string format.
le = LabelEncoder()
healthcare_df['Disease_Presence'] = le.fit_transform(healthcare_df['Disease_Presence'])

In [11]:
# Separate the features and the target variable.
X = healthcare_df.drop('Disease_Presence', axis=1)
y = healthcare_df['Disease_Presence']


y.head()

0    0
1    0
2    1
3    0
4    0
Name: Disease_Presence, dtype: int64

In [27]:
# Step 3: Splitting the Dataset
# Split the dataset into training and testing sets to evaluate the model's performance.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
print("Dataset split into training and testing sets.")

Dataset split into training and testing sets.


In [33]:
# Step 4: Model Building
# Initialize and train a Random Forest Classifier.
clf = RandomForestClassifier(n_estimators=300)
clf.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


In [34]:
# Step 5: Model Evaluation
# Use the test set to make predictions and evaluate the model.
predictions = clf.predict(X_test)
print("Model evaluation results:")
print(classification_report(y_test, predictions))

Model evaluation results:
              precision    recall  f1-score   support

           0       0.49      0.43      0.46      1989
           1       0.50      0.56      0.53      2011

    accuracy                           0.50      4000
   macro avg       0.50      0.50      0.49      4000
weighted avg       0.50      0.50      0.49      4000



In [30]:
 #!pip install pyspark

In [36]:
## Now with Spark MLIB

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Initialize a Spark session
spark = SparkSession.builder.appName("HealthcareMLlib").getOrCreate()

# Load the healthcare dataset with Spark
healthcare_spark_df = spark.read.csv("healthcare.csv", header=True, inferSchema=True)

# Step 1: Preprocessing
# Handle missing values (e.g., fill or drop - here we'll drop for simplicity)
healthcare_spark_df = healthcare_spark_df.na.drop()

# Step 2: Encode categorical target variable if necessary
indexer = StringIndexer(inputCol="Disease_Presence", outputCol="label")
healthcare_spark_df = indexer.fit(healthcare_spark_df).transform(healthcare_spark_df)

# Step 3: Assemble features into a single vector
feature_cols = healthcare_spark_df.columns
feature_cols.remove("Disease_Presence")
feature_cols.remove("label")

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
healthcare_spark_df = assembler.transform(healthcare_spark_df)

# Step 4: Split the data into training and test sets
train_df, test_df = healthcare_spark_df.randomSplit([0.8, 0.2], seed=42)

# Step 5: Apply the Random Forest Classifier
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100)
rf_model = rf.fit(train_df)

# Step 6: Make predictions on the test set
predictions = rf_model.transform(test_df)

# Step 7: Evaluate the model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Model Accuracy: {accuracy}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/15 21:18:30 WARN Utils: Your hostname, Yisraels-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.29 instead (on interface en0)
26/10/15 21:18:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/yisraelwasserman/Projects/dsc400-yis0/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/15 21:18:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/15 21:18:3

Model Accuracy: 0.49817803227485685


26/10/16 04:46:30 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 998856 ms exceeds timeout 120000 ms
26/10/16 04:46:30 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/16 04:46:34 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

# Part 3: Your Turn - Advanced Data Processing and Machine Learning with Spark MLlib

This part of the course challenges you to engage in a comprehensive data processing and machine learning project using Apache Spark. You will apply techniques learned in Part 2 to a new dataset, addressing real-world data complexities such as missing values, anomalies, and the need for advanced aggregations. Additionally, you will implement a machine learning model using Spark MLlib.

## Data Exploration:
- Investigate the 'Telecom Customer Churn' dataset to discover patterns, insights, and anomalies. This step is crucial for understanding the data's structure and content before proceeding with transformations.

## Data Cleaning:
- Address any missing values or format inconsistencies in your dataset. This is essential to ensure the accuracy of your data processing and analysis.

## Feature Engineering:
- Enhance your dataset by creating new features that can provide more depth to your analysis. For example, you might calculate the duration of customer sessions or categorize user activity based on engagement levels.

## Spark SQL:
- Use Spark SQL to perform sophisticated data aggregations that reveal underlying trends. For instance, you might want to analyze the frequency of specific event types per customer or per session.

## Advanced Aggregations:
- Utilize Spark’s capabilities to perform complex aggregations like calculating the average session time per browser type or the total interactions per day.

## Machine Learning with Spark MLlib:
- Apply the knowledge gained in Part 2 to build a machine learning model using Spark MLlib.
- Perform the following steps:
  1. **Data Preparation**: Convert your dataset into a format suitable for machine learning, using techniques such as feature indexing and vector assembly.
  2. **Model Training**: Train a classification model (e.g., RandomForestClassifier) to predict customer churn.
  3. **Model Evaluation**: Evaluate your model using metrics such as accuracy, precision, and recall.

## Visualization and Reporting:
- Create meaningful static and interactive visualizations to represent your findings using Spark or external libraries like `matplotlib` or `plotly`.
- Craft visual representations of your analytical findings to showcase patterns and insights effectively.

## Instructions:
1. Load the 'Telecom Customer Churn Dataset' into Spark and perform initial explorations to understand its structure.
2. Conduct necessary data cleaning and feature engineering to prepare your data for deeper analysis.
3. Use Spark SQL to perform detailed analyses and create relevant aggregations.
4. Build and evaluate a machine learning model using Spark MLlib.
5. Compile your steps, insights, and model results into the Jupyter notebook and submit it as your completed assignment.


In [8]:
# Note: Replace 'telcom_customer_churn.csv' with the actual file path.
telcom_customer_churn_df = pd.read_csv('telecom_customer_churn.csv')